# 第 3 周练习：分词器 + 即时预算分析器

此笔记本比较 Hugging Face 标记器的标记计数，并显示提示消耗了模型上下文窗口的大小。
它还包括一个简单的提示修剪助手，可将提示调整到令牌预算。

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 如果需要，安装依赖项
# !pip -q install transformers sentencepiece



In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 进口
import re
from transformers import AutoTokenizer



In [ ]:
# 比较模型（所有公开）
# 上下文窗口是近似的常见默认值
MODELS = [
    {'name': 'gpt2', 'context': 1024},
    {'name': 'distilbert-base-uncased', 'context': 512},
    {'name': 'bert-base-uncased', 'context': 512},
    {'name': 'google/flan-t5-small', 'context': 512},
]



In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 示例提示（替换为您自己的）
PROMPT = '''
You are a helpful assistant.
Summarize the following text and list 3 action items.

Meeting transcript:
We discussed the Q2 launch plan, timelines, and dependencies.
Engineering will finalize the API integration by next Friday.
Marketing will prepare the announcement draft by Monday.
Support needs a short FAQ for common issues and escalation steps.
Risks include vendor delays and limited QA bandwidth.

Please write a concise summary and three action items.
'''



In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 分词器缓存
_TOKENIZERS = {}

def get_tokenizer(model_name: str):
    if model_name not in _TOKENIZERS:
        _TOKENIZERS[model_name] = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    return _TOKENIZERS[model_name]

def count_tokens(model_name: str, text: str) -> int:
    tok = get_tokenizer(model_name)
    return len(tok.encode(text, add_special_tokens=False))

def budget_report(text: str):
    rows = []
    for m in MODELS:
        n = count_tokens(m['name'], text)
        ctx = m['context']
        pct = round((n / ctx) * 100, 2)
        rows.append({
            'model': m['name'],
            'tokens': n,
            'context': ctx,
            'pct_of_context': pct
        })
    return rows



In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 显示代币预算报告
report = budget_report(PROMPT)
for row in report:
    print(row)



In [ ]:
# 提示修剪：将文本调整到令牌预算中
def trim_to_budget(model_name: str, text: str, max_tokens: int) -> str:
    tok = get_tokenizer(model_name)
    tokens = tok.encode(text, add_special_tokens=False)
    if len(tokens) <= max_tokens:
        return text
    trimmed_tokens = tokens[:max_tokens]
    trimmed_text = tok.decode(trimmed_tokens, skip_special_tokens=True)
    return trimmed_text.rstrip() + '...
'

# 示例：每个模型减少到 80 个标记
for m in MODELS:
    trimmed = trim_to_budget(m['name'], PROMPT, max_tokens=80)
    print('
---', m['name'], '---')
    print(trimmed)

